# Conducting analysis based on data from kaggle ( E-commerce Sales Dataset )

A large volume of data allows you to use a number of tools for analysis, at first glance
 I understand that I will use SQL to sort and combine datasets for more comfortable calculation
 and visualization of important indicators
 Matplotlib and seaborne will be used for preliminary visualization and highlighting
 of the main analytical processes
 Data processing takes place from the point of view of analysis and business goals, not technical use
 (therefore, for each stage of the analysis, I will create separate pieces of code and separate files)
 First of all lets look on data

###  1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sqlite3
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import folium
from folium.plugins import HeatMap

### 2. Data preparation and cleaning

Create function

In [ ]:
def csv_reading(path):
    df = pd.read_csv(path)
    print(df.head(10))
    print('Size of dataFrame: ', df.size)
    print('Type of DataFrame: ', df.dtypes)
    print('Column names: ', df.columns)
    print('Missing values: ',df.isnull().sum())
    print('Duplicates: ',f'{df.duplicated().sum()}/{df.value_counts().sum()} rows')

Use function for customers dataset and make notice

In [ ]:
csv_reading('../content/sample_data/olist_customers_dataset.csv')

That's show column names, type od data, duplicates and missing values (later I can see some
 incorrect values, mistakes Outliers and so on.)
 Firs look on all datasets (geolocation)

In [ ]:
csv_reading('../content/sample_data/olist_geolocation_dataset.csv')

In proces I can understand, why so many duplicates and clean that by some column (id, zip code for example)

In [ ]:
csv_reading('../content/sample_data/olist_orders_dataset.csv')

Some missing values in dates (delivery) - that's not so critic

In [ ]:
csv_reading('../content/sample_data/olist_sellers_dataset.csv')

All is clean and prepare for analyse

In [ ]:
csv_reading('../content/sample_data/olist_products_dataset.csv')

Some missing values about category/photo/name but no duplicates (id is unique) that's why i cant
 make some replace or changing with frequency or mean

In [ ]:
csv_reading('../content/sample_data/olist_order_items_dataset.csv')

In [ ]:
csv_reading('../content/sample_data/olist_order_payments_dataset.csv')

csv_reading('../content/sample_data/olist_order_reviews_dataset.csv')
 reviews comments and messages are not so important for analyse ( Score much more important )

# 3. Exploatory Analysis

Download CSV files

In [ ]:
orders = pd.read_csv('../content/sample_data/olist_orders_dataset.csv')
items = pd.read_csv('../content/sample_data/olist_order_items_dataset.csv')
payments = pd.read_csv('../content/sample_data/olist_order_payments_dataset.csv')
products = pd.read_csv('../content/sample_data/olist_products_dataset.csv')
cat_name = pd.read_csv('../content/sample_data/product_category_name_translation.csv')

Create SQL Base

In [ ]:
conn = sqlite3.connect('../content/sample_data/Ex_An.db')

And from csv files into tables

In [ ]:
orders.to_sql('orders', conn, if_exists='replace', index = False)
items.to_sql('items', conn, if_exists='replace', index = False)
payments.to_sql('payments', conn, if_exists='replace', index = False)
products.to_sql('products', conn, if_exists='replace',index = False)
cat_name.to_sql('names', conn, if_exists='replace',index = False)

SQL work and preparing and firs analyse in Sales (count of orders by category or product)

In [ ]:
query = ("""
SELECT
    o.order_id,
    i.freight_value,
    s.product_id,
    p.payment_type,
    n.product_category_name_english,
    i.price,
    p.payment_value
FROM orders o
JOIN items i ON o.order_id = i.order_id
JOIN payments p ON o.order_id = p.order_id
Join products s ON i.product_id = s.product_id
JOIN names n ON s.product_category_name = n.product_category_name;
""")

In [ ]:
df = pd.read_sql_query(query, conn)
print(df.head())
print(df.columns)

Make EDA analysis

In [ ]:
print(df.describe())

1. Gross Revenue ( Sum of prices ) without Freight Revenue - that is logistic

In [ ]:
print('Gross Revenue = ', sum(df['price']))

2. Gross Freight Revenue:

In [ ]:
print('Gross Freight Revenue = ', sum(df['freight_value']))

3. Orders count (how much orders)

In [ ]:
print('Orders count: ', len(df['order_id'].unique()))

4. Items sold

In [ ]:
print('Items sold: ', len(df['product_id']))

5. Average Check price

In [ ]:
print('Average Check price: ', sum(df['price'])/len(df['order_id'].unique()))

6. Price Distribution (min, max, median and average price + visualization binning)

In [ ]:
medianprice = df['price'].median()
print(medianprice)
plt.figure(figsize = (10,7))
plt.boxplot(df['price'])
plt.title('Price analysis')
plt.show()

So median price is 74.9 but in data are many emissions,
 that should be grouped and look at the number of orders by group and the cost of prices

Segments ( low , Medium , Height , Lux ) prices

In [ ]:
bins = [df['price'].min(), 100, 500, 2500, df['price'].max()]
labels = ['low', 'medium', 'height', 'lux']
df['price_seg'] = pd.cut(df['price'], bins= bins, labels = labels, include_lowest=True)
s_counts = df['price_seg'].value_counts().sort_index()
plt.figure(figsize=(10,7))
s_counts.plot(kind='bar')
plt.title('Count of products by price segmentation')
plt.xlabel('Price segmentation')
plt.ylabel('Count of products')
plt.show()

Let's find most soldet Products by count of soldet items and by sales revenue

In [ ]:
m_sold = df.groupby('product_category_name_english')['price'].sum()
m_sold_sorted = m_sold.sort_values()
print(m_sold_sorted.tail(10))
plt.figure(figsize = (10, 7))
m_sold_sorted.tail(10).plot(kind='bar')
plt.xlabel('Category')
plt.ylabel('Sales Revenue')
plt.title('Top soldet category of products sorted by sales revenue')
plt.show()

As a result, we see the top sold products according to the income received from sales
(we can say that these products are the most in demand on the market, but this is not the final estimate)
The initial analysis allows you to only evaluate the picture and continue the research


In [ ]:
products_counts = df['product_category_name_english'].value_counts()
print(products_counts.head(10))
products_counts.head(10).plot(kind='bar')
plt.xlabel('Product category')
plt.ylabel('Count of orders')
plt.title('Top soldet category of products sorted by count of orders')
plt.show()

therefore, it is basically possible to draw conclusions about the most popular products on the market,
according to the number of products ordered and the profit received

From the point of view of financial analytics, I am interested in the payment methods and the
average check for each payment method

In [ ]:
pay = df['payment_type'].value_counts()
print(pay)

let's build pie chart for visualization

In [ ]:
plt.pie(pay, labels=pay.index, autopct='%1.1f%%')
plt.title('Payment method')
plt.show()

Credit cart payment method is on first place with 73% of total payment methods
Average check for each payment method

In [ ]:
avg_pay = df.groupby('payment_type')['price'].mean()
print(avg_pay)

## that's give some notice for analyze in future

# 4. Time Analyse

I want to work with time dataset in order to understand the patterns of peak orders and the
differentiation of customer activity, the data of the study allow more productive use of marketing
tools to improve one's position in the market and efficient use of resources

First of all preparing date for analysis (take years/months/days/hours of orders) change that into datetime
Lets use SQL for collecting important datasets in this part of analysis
I need order id , order purchase, time deliver, date payment value.

In [ ]:
orders = pd.read_csv('../content/sample_data/olist_orders_dataset.csv')
payments = pd.read_csv('../content/sample_data/olist_order_payments_dataset.csv')

conn = sqlite3.connect('../content/sample_data/Time.db')
orders.to_sql('orders',conn, if_exists='replace', index=False)
payments.to_sql('payments',conn, if_exists='replace', index=False)

query= """
        Select
            o.order_id,
            o.order_purchase_timestamp,
            o.order_approved_at,
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date,
            p.payment_value
        From orders AS o
        Join payments p ON o.order_id = p.order_id;
            """
time_df = pd.read_sql_query(query,conn)

and check dataset

In [ ]:
print(time_df.head(10))

Make datetime sets

In [ ]:
time_df['order_purchase_timestamp'] = pd.to_datetime(time_df['order_purchase_timestamp'])
time_df['order_approved_at'] = pd.to_datetime(time_df['order_approved_at'])
time_df['order_delivered_customer_date'] = pd.to_datetime(time_df['order_delivered_customer_date'])
time_df['order_estimated_delivery_date'] = pd.to_datetime(time_df['order_estimated_delivery_date'])

Separate time into (years, months, days, hours) order purchase and analyse

In [ ]:
time_df['year'] = time_df['order_purchase_timestamp'].dt.year
time_df['month'] = time_df['order_purchase_timestamp'].dt.month
time_df['day_o_w'] = time_df['order_purchase_timestamp'].dt.day_name()
time_df['hour'] = time_df['order_purchase_timestamp'].dt.hour

Let's see orders sorted by date(years and months) for understanding peaks of orders

In [ ]:
order_month_year = time_df.groupby(['year','month'])['order_id'].count()
order_month_year.plot(kind='line', figsize=(15,7),title='Orders by month and year')
plt.show()
order_month = time_df.groupby('month')['order_id'].count()
order_month.plot(kind='line', figsize=(15,7),title='Orders by month')
plt.show()

analysis separately by year and month gives a general picture of peak orders, which increase in the summer period if taken in general by month and directly in the 11th month (November), possibly connected with "Black Friday". A strong decline in the autumn period may indicate a lack of interest from buyers or a lack of profitable offers from sellers.


 Orders sorted by days of week

In [ ]:
order_da = time_df.groupby('day_o_w')['order_id'].count().sort_values()
order_da.plot(kind='bar', figsize=(10,7),title='Orders by days')
plt.show()

Most orders are placed at the beginning of the week from Monday to Wednesday (this may be due to the fact that customers want to receive the product on the weekend, which will allow to be present at the delivery or for other reasons) a short note - a marketing strategy on the weekend stimulates the incentive to buy and gives results


Orders sorted by hours

In [ ]:
order_hour = time_df.groupby('hour')['order_id'].count().sort_values()
order_hour.plot(kind='bar', figsize=(10,7),title='Orders by hours')
plt.show()

Buyer activity starts at 11 a.m. and peaks during this time until at least 11 p.m. But much better make some bins such like (morning, day, evening, night)


In [ ]:
day_bins = [0,6,12,17,24]
labels_d = ['Night (0-6)','Morning (6-12)','Afternoon (12-18)','Evening (18-24)']
time_df['time_of_day']=pd.cut(time_df['hour'], bins=day_bins, labels=labels_d, include_lowest=True,right=False)
time_df['time_of_day'].value_counts().sort_values().plot(kind='bar',figsize = (10,7), title='Orders by time of day')
plt.show()

Naturally, most orders are made in the evening from 5 to 11 p.m. (free time after work)
Combo : make heatmap and see orders sorted by day of week and hours
Create pivot table

In [ ]:
pivot = time_df.pivot_table(index = 'day_o_w',columns = 'time_of_day', values= 'order_id',aggfunc='count',fill_value=0)
order_days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex(order_days)
plt.figure(figsize = (8,11))
plt.imshow(pivot)
plt.xticks(range(len(pivot.columns)),pivot.columns,rotation = 25)
plt.yticks(range(len(pivot.index)),pivot.index)
plt.colorbar(label='Number of Orders')
plt.title('Orders by day of week and hours')
plt.show()

Now I make time analyse for delivery time and make some insights proportion of delays. I need clean data without info about canceled orders order not completed


In [ ]:
df_c_time = time_df.dropna(subset=['order_delivered_customer_date','order_purchase_timestamp','order_estimated_delivery_date']).copy()
print(df_c_time.head(10))

Change metrics in datetime (days):
How much day need from purchase date to delivery date - that give some info about logistic


In [ ]:
df_c_time.loc[:, 'delivery_days'] = (df_c_time['order_delivered_customer_date']-df_c_time['order_purchase_timestamp']).dt.days
average_delivery = df_c_time['delivery_days'].mean()
print(f'Average delivery time - {average_delivery} days')

what is the difference between the expectation and the actual delivery, which will allow you to evaluate the speed and quality of logistics services ('+' value will be positive, '-' will be negative for logistics evaluation)


In [ ]:
df_c_time.loc[:,'estimate_time'] = (df_c_time['order_estimated_delivery_date'] - df_c_time['order_purchase_timestamp']).dt.days
df_c_time.loc[:,'faster_days'] = (df_c_time['order_estimated_delivery_date']-df_c_time['order_delivered_customer_date']).dt.days
df_c_time.loc[:,'delay_days'] = (df_c_time['order_delivered_customer_date'] - df_c_time['order_estimated_delivery_date']).dt.days
plt.figure(figsize=(10,7))
plt.boxplot(df_c_time['delivery_days'].dropna(),vert=False)
plt.title('Distribution of Delivery Time')
plt.xlabel('Days')
plt.show()

time is different

In [ ]:
avg_faster_del = (df_c_time['estimate_time']-df_c_time['faster_days']).mean()
print(f'Faster delivery average - {avg_faster_del} days')

Percentage of delivery efficient

In [ ]:
p_f_d = ((df_c_time['estimate_time']-df_c_time['delivery_days'])/df_c_time['estimate_time']) * 100
print(f'Percentage of delivery efficient - {p_f_d.mean()} %')
counts = []
for late in df_c_time['delay_days']:
    if late > 0 :
        counts.append(late)
late_count = len(counts)
print(f'Late delivery count - {late_count} orders/{df_c_time['delay_days'].value_counts().sum()} orders')

proportion of delays

In [ ]:
late_deliveries = (df_c_time['delay_days'] > 0).mean() * 100
print(f'proportion of delays - {late_deliveries} %')

visualization histogram for showing how often orders arrive ahead of schedule

In [ ]:
plt.figure(figsize=(10,8))
df_c_time['faster_days'].hist(bins = 10, edgecolor = 'black')
plt.title('Distribution of faster delivery')
plt.xlabel('Days faster as plan')
plt.ylabel('Count of orders')
plt.show()

### Insights :

1.Order activity is recorded according to the season (this may be related to holidays, promotions, and advantageous offers)
2. Most orders are placed in the evening (5:00 PM–11:00 PM), which indicates customer activity after work. This can be a key time for targeted promotions and marketing campaigns.
3. The average faster delivery time was about 12 days, which is in most cases less than the expected time. This indicates good logistics efficiency.
4.On average, orders arrive 12 days faster than predicted, which can increase customer satisfaction and increase their loyalty. Percentage of delivery efficient - 47.23 %
5. The proportion of delays is 6.73 %, meaning the company consistently adheres to the stated delivery times. This is a competitive advantage.
6. Despite the overall positive picture, it is worth analyzing exceptional cases with long deliveries
to understand their reasons (geography, product category, seasonal peaks).


# 5. Rating Analyse

In [ ]:
reviews = pd.read_csv('../content/sample_data/olist_order_reviews_dataset.csv')
cat_name = pd.read_csv('../content/sample_data/product_category_name_translation.csv')
products = pd.read_csv('../content/sample_data/olist_products_dataset.csv')
items = pd.read_csv('../content/sample_data/olist_order_items_dataset.csv')
conn = sqlite3.connect('../content/sample_data/review_r.db')
reviews.to_sql('reviews',conn, index=False, if_exists='replace')
cat_name.to_sql('names', conn, if_exists='replace',index = False)
products.to_sql('products', conn, if_exists='replace',index = False)
items.to_sql('items',conn, index= False, if_exists='replace')
query_rev = """
Select
    review_score, COUNT(*) as count_reviews
FROM reviews
GROUP BY review_score
ORDER BY review_score;
"""
df_reviews = pd.read_sql_query(query_rev, conn)
print(df_reviews.head(10))

plt.figure(figsize=(10,7))
labels = {'Very bad':1,'Bad': 2,'Normal':3,'Good':4,'Very good':5}
plt.pie(x = df_reviews['review_score'],labels=labels , autopct='%1.1f%%')
plt.title('Percentage of reviews')
plt.show()

Most customer reviews are positive, with 33.3% rating 5 (very good), 26.7% rating 4 (good), and only 20% (6.7% very bad and 13.3% bad) having negative reviews. Overall, these are positive indicators, it is worth exploring these metrics more deeply and finding out what the patterns are.


In [ ]:
query_rev_b_p = """
Select
    COUNT(*) as number_of_reviews,
    AVG(r.review_score) AS avg_review_score,
    n.product_category_name_english
FROM items as i
JOIN products p ON i.product_id = p.product_id
Join reviews as r ON i.order_id = r.order_id
JOIN names n ON p.product_category_name = n.product_category_name
GROUP BY n.product_category_name_english
Order by avg_review_score DESC;
"""

df_rev_avg = pd.read_sql_query(query_rev_b_p,conn)
print(df_rev_avg.head(10))

Visualization for Top 10 height product categories by rating (reviews) and Top 10 low rating

In [ ]:
df_rev_avg.head(10).plot(kind='bar', x = 'product_category_name_english' , y ='avg_review_score', legend = False)
plt.title ('Top 10 categories of products by height rating (reviews)')
plt.ylabel('Average review score')
plt.xticks(rotation = 45, ha = 'right')
plt.show()

In [ ]:
df_rev_avg.tail(10).plot(kind = 'bar', x = 'product_category_name_english' , y ='avg_review_score',color = 'red', legend = False)
plt.xticks(rotation = 90)
plt.ylabel('Average review score')
plt.title('Top 10 categories of products by low rating (reviews)')
plt.show()

Based on the visualization of the results, it is possible to distinguish categories of goods with an average low or, on the contrary, a high rating (searches and ratings of customers), this makes it possible to identify problems and find the reasons for low ratings)


in more detail, you can look at product ratings directly by the number of reviews on ratings for each category.

In [ ]:
query_disc = """
SELECT
    COUNT(*) as counts,
    r.review_score,
    n.product_category_name_english
FROM items as i
JOIN products p ON i.product_id = p.product_id
Join reviews as r ON i.order_id = r.order_id
JOIN names n ON p.product_category_name = n.product_category_name
GROUP BY n.product_category_name_english, review_score
Order by n.product_category_name_english, review_score;
"""
df_rev_disc = pd.read_sql_query(query_disc,conn)
print(df_rev_disc.head(10))

change table (pivot)

In [ ]:
pivot_rev = df_rev_disc.pivot(index='product_category_name_english', columns = 'review_score', values='counts').fillna(0)
top_category = pivot_rev.sum(axis=1).sort_values(ascending=False).head(10).index
pivot_top = pivot_rev.loc[top_category]
pivot_top.plot(kind='bar',stacked = True, figsize = (10,7))
plt.title('Reviews of top 10 categories')
plt.xticks(rotation = 45)
plt.ylabel('Number of reviews')
plt.xlabel('Category')
plt.show()

taking into account reviews and sorting by ratings of the top categories, you can observe a positive trend and many good customer ratings. In addition, it is possible to observe that the ratings are mostly positive (4-5) or negative (1), because there are very few ratings (2-3) compared to the general


It is also interesting to look at reviews and messages from customers. Since the reviews are in Portuguese, we will use a translator and find approximately words that can describe the product or be contained in the reviews in order to sort positive and negative comments (words like "good", "terrible", "delay", "satisfied", "long", "defect", "recommend")


In [ ]:
query_kom = """
SELECT
    SUM(CASE WHEN review_comment_message LIKE '%otimo%'
        OR review_comment_message LIKE '%excelente'
        OR review_comment_message LIKE '%bom%' THEN 1 ELSE 0 END) AS positive_commentaries,
    SUM(CASE WHEN review_comment_message LIKE '%ruim%'
        OR review_comment_message LIKE '%pessimo%'
        OR review_comment_message LIKE '%atraso%' THEN 1 ELSE 0 END) AS negative_commentaries
FROM reviews
"""
df_rev_kom = pd.read_sql_query(query_kom,conn)
print(df_rev_kom)


In [ ]:
df_review_delivery = pd.merge(df_c_time, reviews, on = 'order_id', how = 'inner')
df_review_delivery = df_review_delivery.dropna(subset=['order_delivered_customer_date', 'review_score'])
avg_score_by_delay = df_review_delivery.groupby('delay_days')['review_score'].mean()
print(avg_score_by_delay)

In [ ]:
avg_score_by_delay.plot(marker = 'o')
plt.title('Dependence of rating on delivery delay')
plt.xlabel('Count of Days of delay')
plt.ylabel('Average review_score')
plt.show()

Looking at the graph, it is immediately clear how the rating and ratings from customers fall depending on the delay


In [ ]:
correlation = df_review_delivery['review_score'].corr(df_review_delivery['delay_days'])
print(f'Correlation between delay and reviews rating: {correlation}')

-0.26 correlation index means that there is still a dependence of the assessment on the speed of delivery and delay, although it is not high enough (this may be due to good evaluations from customers not depending on the speed of delivery)

# Insights

1. Most buyers leave positive reviews — the average rating is above 4 points, which indicates overall satisfaction with the service.
2. At the same time, there are categories with significantly lower average ratings, which may signal problems with the quality of goods or service in these segments.
3. Text analysis of reviews showed that buyers most often use positive formulations such as “ótimo”, “excelente”, but negative mentions (“ruim”, “atraso”) are often related to delivery issues.
4. Preliminary analysis indicates a possible relationship between delivery delay and low ratings — this factor is worth checking in more detail.

# 6. Demografic Analyse

In [ ]:
geo = pd.read_csv('../content/sample_data/olist_geolocation_dataset.csv')
cust = pd.read_csv('../content/sample_data/olist_customers_dataset.csv')
items = pd.read_csv('../content/sample_data/olist_order_items_dataset.csv')
product = pd.read_csv('../content/sample_data/olist_products_dataset.csv')
names = pd.read_csv('../content/sample_data/product_category_name_translation.csv')
orders = pd.read_csv('../content/sample_data/olist_orders_dataset.csv')

conn = sqlite3.connect('../content/sample_data/dem_an.db')
orders.to_sql('orders', conn, index=False, if_exists='replace')
product.to_sql('product',conn,if_exists='replace',index = False)
geo.to_sql('geo',conn, index=False, if_exists='replace')
cust.to_sql('cust',conn,index = False, if_exists='replace')
items.to_sql('items',conn, index=False, if_exists='replace')
names.to_sql('names',conn, index= False, if_exists='replace')

query_dem1 = """Select customer_state, customer_city, customer_id, customer_unique_id from cust"""
df_cust = pd.read_sql_query(query_dem1, conn)
print(df_cust.head(10))

calculate number of customers by states and city

In [ ]:
count_cust_city = df_cust.groupby('customer_city')['customer_id'].count().sort_values()
print(count_cust_city)

Visualization for top 20 Cities sorted by count of customers


In [ ]:
count_cust_city.tail(20).plot(kind='bar')
plt.title('top 20 Cities sorted by count of customers')
plt.xlabel('City')
plt.ylabel('Count of customers')
plt.show()

curitiba, brasilia, belo horizonte, rio de janeiro, sao paulo - when analyzing the number of orders according to the list, these cities are the concentration of the largest number of orders, such information allows sellers to understand the need to place warehouses (logistics nodes) in order to improve and speed up the possibilities of realization

same thing for states

In [ ]:
count_cust_state = df_cust.groupby('customer_state')['customer_id'].count().sort_values()
print(count_cust_state)
count_cust_state.plot(kind='bar', color = 'red')
plt.title('States sorted by count of customers')
plt.xlabel('State')
plt.ylabel('Count of customers')
plt.show()

use folium library for geolocation and concentration of orders visualization


In [ ]:
query_map = """
SELECT c.customer_id, g.lat, g.lng From cust c
Join (SELECT geolocation_city, AVG(geolocation_lat) as lat, AVG(geolocation_lng) as lng FROM geo
Group BY geolocation_city) g ON c.customer_city = g.geolocation_city; """
df_map = pd.read_sql_query(query_map, conn)
print(df_map.head(10))
br_center = folium.Map(location=[-15.78, -48.93], zoom_start = 4)
HeatMap(df_map[['lat','lng']].values, radius=10).add_to(br_center)

Save that map in html file than we can see that in browser


In [ ]:
br_center.save('../content/sample_data/customer_heatmap.html')

braz_map = folium.Map(location = [-15.78, 47.93], zoom_start=4)
for _, row in df_map.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius = 2,
        color = 'red',
        fill = True,
        fill_opacity = 0.5).add_to(braz_map)
braz_map.save('../content/sample_data/braz_map.html')

In [ ]:
braz_map

In [ ]:
br_center

analysis of customer behavior (repeatability)

In [ ]:
query_cust = """
SELECT c.customer_unique_id, COUNT(o.order_id) as orders_count
FROM orders AS o
JOIN cust c ON o.customer_id = c.customer_id
Group by c.customer_unique_id;"""
df_o_c = pd.read_sql_query(query_cust,conn)

In [ ]:
df_o_c['repeat_customer'] = df_o_c['orders_count'] > 1
repeat_rate = df_o_c['repeat_customer'].mean() * 100
print(f'Percentage repeat customers: {repeat_rate}')
df_o_c['orders_count'].value_counts().sort_index().plot(kind = 'hist')
plt.title('Count of orders per customer')
plt.xlabel('Count of orders')
plt.ylabel('Count of customers')
plt.show()

In [ ]:
df_items_per_order = items.groupby('order_id')['product_id'].count().reset_index()
print(f'Average count of products in orders: {df_items_per_order["product_id"].mean()}')

In [ ]:
query_orders_cust = """
SELECT o.order_id, c.customer_unique_id
FROM orders o
JOIN cust c ON o.customer_id = c.customer_id;
"""
orders_cust = pd.read_sql_query(query_orders_cust, conn)


repeat customers

In [ ]:
repeat_customers = df_o_c[df_o_c['orders_count'] > 1]['customer_unique_id']
repeat_orders = orders_cust[orders_cust['customer_unique_id'].isin(repeat_customers)][['order_id','customer_unique_id']].reset_index(drop=True)
repeat_items = items[items['order_id'].isin(repeat_orders['order_id'])]
top_repeat_products = repeat_items['product_id'].value_counts().head(10)
print(f'Repeat orders with id of product : {top_repeat_products}')

# 7. Sellers Analyse

In [ ]:
sellers = pd.read_csv('../content/sample_data/olist_sellers_dataset.csv')
product = pd.read_csv('../content/sample_data/olist_products_dataset.csv')
items = pd.read_csv('../content/sample_data/olist_order_items_dataset.csv')
names = pd.read_csv('../content/sample_data/product_category_name_translation.csv')
conn = sqlite3.connect('../content/sample_data/salers.db')
sellers.to_sql('sellers',conn, if_exists='replace',index=False)
items.to_sql('items',conn, if_exists='replace',index = False)
product.to_sql('product',conn,if_exists='replace',index = False)
names.to_sql('names',conn, if_exists='replace',index = False)

query_sellers = """SELECT s.seller_id, s.seller_city, s.seller_state, i.price, i.order_id, p.product_id, n.product_category_name_english FROM sellers AS s
Join items AS i ON i.seller_id = s.seller_id
JOIN product AS p ON p.product_id = i.product_id
JOIN names AS n ON n.product_category_name = p.product_category_name;"""

df_sellers = pd.read_sql_query(query_sellers,conn)
print(df_sellers.head(10))

top 10 sellers

In [ ]:
top_10_sellers = df_sellers['seller_id'].value_counts()
print(top_10_sellers.head(10))

and show info with barchart for top sellers

In [ ]:
top_10_sellers.head(10).plot(kind= 'bar')
plt.title('Top sellers id sorted by count of sales (orders)')
plt.xlabel('seller ID')
plt.ylabel('Count of orders')
plt.show()

and show top sellers sorted by Revenue

In [ ]:
total_sales = df_sellers.groupby('seller_id')['price'].sum().sort_values()
top_10_sellers = total_sales.tail(10)
print(top_10_sellers)
top_10_sellers.plot(kind = 'bar', color = 'gold')
plt.title('Top sellers id sorted by Revenue (price sum)')
plt.xlabel('Sellers ID')
plt.ylabel('Revenue')
plt.show()

Concentration of top sellers in Total revenue in market

In [ ]:
other = total_sales.sum() - top_10_sellers.tail(10).sum()
seller_share = top_10_sellers._append(pd.Series({'Others' : other}))

plt.figure(figsize=(8,6))
plt.pie(seller_share, labels = seller_share.index, autopct='%1.1f%%')
plt.title('Sales part of the top 10 sellers on Market')
plt.show()

Geo position of sellers , where are the most of sellers(city/ state)

In [ ]:
geopos = df_sellers.groupby('seller_city')['seller_id'].nunique().sort_values()
print(geopos.tail(10))

Show by barchart

In [ ]:
geopos.tail(10).plot(kind = 'bar',color = 'green')
plt.title('Top 10 cities sorted by count of sellers')
plt.xlabel('City')
plt.ylabel('Count of sellers')
plt.show()

Same thing for state of sellers

In [ ]:
geoposstate = df_sellers.groupby('seller_state')['seller_id'].nunique().sort_values(ascending=False)
print(geoposstate.head(10))

Show that with bar chart

In [ ]:
geoposstate.head(10).plot(kind='bar', color = 'brown')
plt.title('Top 10 states sorted by count of sellers')
plt.xlabel('State')
plt.ylabel('Count of sellers')
plt.show()

Top products distributed among the top 10 sellers (find out what the top sellers sell)

In [ ]:
top_10_sellers = items.groupby('seller_id')['price'].sum().sort_values(ascending=False).head(10)
top_sellers_id = top_10_sellers.index
items_products = items.merge(product, on = 'product_id', how = 'left')
items_products = items_products.merge(names, on = 'product_category_name', how = 'left')
top_seller_items = items_products[items_products['seller_id'].isin(top_sellers_id)]
top_product = (top_seller_items.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False).head(10))
print(top_product)
top_product.plot(kind='bar',color = 'red')
plt.title('Top 10 categories of products  in Top 10 sellers')
plt.xlabel('Product category')
plt.ylabel('Count of sales')
plt.show()

# Some insights

1 Given the distribution and frequent top 10 sellers from the total revenue, it can be said that the market and income are distributed approximately equally (which indicates the absence of monopolists)
The pie chart showed that the top 10 sellers cover only 12.3% of the total sales market
2 The best-selling categories among the top sellers are (watches_gifts,bed_bath_tableoffice_furniture, furniture_decor, computers, cool_stuff, telephony, housewares, health_beauty,home_comfort)
Which makes it possible to understand what the top sellers are betting on
3 Concentration and sales, which are displayed according to demographic indicators (geo-positioning), make it possible to understand the concentration and distribution of sales among cities and states - this makes it possible to study possible competition and a free niche for creating trade

Having made certain small conclusions, it is possible to summarize as a result (which territories should be explored by newcomers in the online sales sector, which products are in demand, taking into account the experience of top sellers, taking into account the fact that the market adheres to effective competition due to the absence of monopolists)

Note: you can analyze the reviews according to the sellers, determine the rating of the tops, whether the positive reviews on the tops prevail or vice versa. This will give a good picture of the interdependence of gross income and reputation

# 8. Future Prediction

In [ ]:
orders = pd.read_csv('../content/sample_data/olist_orders_dataset.csv')
items = pd.read_csv('../content/sample_data/olist_order_items_dataset.csv')
names = pd.read_csv('../content/sample_data/product_category_name_translation.csv')
conn = sqlite3.connect('../content/sample_data/predict.db')
items.to_sql('items',conn, if_exists='replace',index = False)
orders.to_sql('orders', conn, index=False, if_exists='replace')
names.to_sql('names',conn, if_exists='replace',index = False)

query_p = """
SELECT o.order_id, o.order_purchase_timestamp,i.price, i.freight_value
FROM orders AS o
Join items AS i ON o.order_id = i.order_id;"""

In [ ]:
df_predict = pd.read_sql_query(query_p, conn)
df_predict['order_purchase_timestamp'] = pd.to_datetime(df_predict['order_purchase_timestamp'])
df_predict['revenue'] = df_predict['price'] + df_predict['freight_value']
df_predict['month'] = df_predict['order_purchase_timestamp'].dt.to_period('M')

monthly_revenue = df_predict.groupby('month')['revenue'].sum().reset_index()
monthly_revenue['month'] = monthly_revenue['month'].dt.to_timestamp()
monthly_revenue['month_num'] = np.arange(len(monthly_revenue))

X = monthly_revenue[['month_num']]
y = monthly_revenue['revenue']
# Linear Regression predict
model = LinearRegression()
model.fit(X, y)
y_predict = model.predict(X)

Metrix quality

In [ ]:
r2 = r2_score(y, y_predict)
mae = mean_absolute_error(y, y_predict)
rmse = np.sqrt(mean_squared_error(y, y_predict))

print(f'R2 : {r2}')
print(f'MAE : {mae}')
print(f'RMSE : {rmse}')

R^2 how good model shows variability of data ( 0.504 ) normal the linear model shows the basic trend but does not take into account all variations (seasonality, promotions, large orders)
MAE ( Mean Absolute Error ) gives an idea of the average error of the forecast (how much, on average, the model lags or leads the actual income) 175 thousand is a rather large error, but it can be acceptable for an approximate forecast
RMSE - mean squared Error ( sensible to big deviation ) highlights the impact of abnormally high or low sales (model accuracy)
The model shows the general trend well, but individual peak months are predicted less accurately


Predict for next 6 month

In [ ]:
future_month = np.arange(len(monthly_revenue), len(monthly_revenue) + 6).reshape(-1,1)
future_preds = model.predict(future_month)
future_dates = pd.date_range(start=monthly_revenue['month'].iloc[-1] + pd.offsets.MonthBegin(1), periods = 6, freq = 'MS')
forecast_df = pd.DataFrame({'month': future_dates, 'revenue' : future_preds})

plt.figure(figsize = (12,8))
plt.plot(monthly_revenue['month'], monthly_revenue['revenue'], label = 'Fact Data', marker = 'o')
plt.plot(monthly_revenue['month'], y_predict, label = 'Linear Regression', color = 'red')
plt.plot(forecast_df['month'], forecast_df['revenue'], color = 'green', linestyle = '--',marker = 'o', label = 'predict')
plt.title('Predict of revenue (Linear Regression)')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.legend()
plt.grid(True)
plt.show()

According to the visualization, it is clear that in the future period (6 months) the income will grow (based on forecasting and the linear regression model)


In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['month'] = orders['order_purchase_timestamp'].dt.to_period('M')

monthly_order = orders.groupby('month')['order_id'].count().reset_index()
monthly_order['month']= monthly_order['month'].dt.to_timestamp()
monthly_order['month_num'] = np.arange(len(monthly_order))

X1 = monthly_order[['month_num']]
y1 = monthly_order['order_id']

model = LinearRegression()
model.fit(X1, y1)
y_pred = model.predict(X1)

r2o = r2_score(y1, y_pred)
mae_o = mean_absolute_error(y1, y_pred)
rmse_o = np.sqrt(mean_squared_error(y1, y_pred))

print(f'R2 orders : {r2o}')
print(f'MAE orders : {mae_o}')
print(f'RMSE orders : {rmse_o}')

In [ ]:
future_months = np.arange(len(monthly_order), len(monthly_order)+6).reshape(-1,1)
future_pred = model.predict(future_months)
future_dates = pd.date_range(start = monthly_order['month'].iloc[-1] + pd.offsets.MonthBegin(1), periods = 6, freq='MS')
future_df  = pd.DataFrame({'month' : future_dates, 'orders' : future_pred})

plt.figure(figsize=(8,6))
plt.plot(monthly_order['month'], monthly_order['order_id'], label = 'Fact (count of orders)', marker = 'x')
plt.plot(monthly_order['month'],y_pred, label = 'Regression line', linestyle = '--')
plt.plot(future_df['month'], future_df['orders'],label = 'Predict 6 month', marker = 'o', color = 'red')
plt.xlabel('Month')
plt.title('Prognose (count of orders) Linear Regression')
plt.ylabel('Count orders')
plt.show()

In [ ]:
df = orders.merge(items, on="order_id", how="inner")
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

Income sorted by orders

In [ ]:
df['revenue'] = df['price'] + df['freight_value']

include month

In [ ]:
df['month'] = df['order_purchase_timestamp'].dt.to_period('M')

Average chek group by month

In [ ]:
monthly_aov = df.groupby('month').agg({'revenue':'sum','order_id':'nunique'}).reset_index()
monthly_aov['AOV'] = monthly_aov['revenue'] / monthly_aov['order_id']
monthly_aov['month'] = monthly_aov['month'].dt.to_timestamp()
monthly_aov['month_num'] = np.arange(len(monthly_aov))
X = monthly_aov[['month_num']]
y = monthly_aov['AOV']
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f'R2 : {r2}')
print(f'MAE : {mae}')
print(f'RMSE : {rmse}')

Predict for average chek

In [ ]:
future_months = np.arange(len(monthly_aov), len(monthly_aov)+6).reshape(-1,1)
future_preds = model.predict(future_months)
future_dates = pd.date_range(start=monthly_aov['month'].iloc[-1] + pd.offsets.MonthBegin(1), periods=6, freq='MS')
forecast_df = pd.DataFrame({'month': future_dates, 'AOV': future_preds})

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(monthly_aov['month'], monthly_aov['AOV'], label='Fact (AOV)', marker='o')
plt.plot(monthly_aov['month'], y_pred, label='Linear Regression', linestyle='--')
plt.plot(forecast_df['month'], forecast_df['AOV'], label='Predict (6 month)', marker='x', color='red')
plt.title('Predict for average check  (AOV) (Linear Regression)')
plt.xlabel('Month')
plt.ylabel('AOV (average check)')
plt.grid(True)
plt.show()
print(forecast_df)

The value of p2 is close to zero (0.08), this means that the model covers only 8% of the data, so using linear regression prediction is not effective, it is easier to use basic mean prediction here


In [ ]:
monthly_avg_ticket = df_predict.groupby('month')['revenue'].mean().reset_index()
monthly_avg_ticket['month']= monthly_avg_ticket['month'].dt.to_timestamp()

Basic predict - mean value

In [ ]:
avg_value = monthly_avg_ticket['revenue'].mean()

predict for 6 month

In [ ]:
future_dates = pd.date_range(start=monthly_avg_ticket['month'].iloc[-1] + pd.offsets.MonthBegin(1), periods=6, freq='MS')
forecast_df = pd.DataFrame({'month': future_dates,'avg_ticket_forecast': [avg_value] * 6})

print("Mean value:", avg_value)
print(forecast_df)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(monthly_avg_ticket['month'], monthly_avg_ticket['revenue'], label='Fact')
plt.hlines(avg_value, monthly_avg_ticket['month'].min(), future_dates[-1], colors='red', linestyles='dashed', label='Mean (predict)')
plt.scatter(forecast_df['month'], forecast_df['avg_ticket_forecast'], color='orange', label='Predict')
plt.title('Predict average check')
plt.xlabel('Month')
plt.ylabel('Average check')
plt.legend()
plt.show()

Average check remains almost at the same level of 135-140, this indicates stability, a simple noticeable gradual decrease in the average value of a check (this can be influenced by many factors, such as the economic situation and the ability of customers to pay, as well as the economic situation directly in the country and in the world)

Analysis and forecasting of basic indicators (revenue, number of orders and average check) allows companies to formulate the right approach in terms of marketing, logistics, and choice of strategies. In the future, it is possible to analyze products that will be in the greatest demand in the future and for which demand is growing.